# AI Property Due-Diligence Analyzer

## Business Problem

Real estate investors must perform due-diligence before purchasing properties.  
This process involves evaluating financial, legal, structural, and neighborhood risks.

However, real estate datasets containing due-diligence reports are difficult to access
because they contain sensitive financial and legal information.

## Solution

This project builds an AI-powered synthetic dataset generator that creates
realistic **property due-diligence reports** using multiple language models
and prompt engineering strategies.

The generated dataset can be used for:

- training AI property analysis models
- testing investment risk classifiers
- experimenting with synthetic real-estate datasets

The application includes an interactive **Gradio UI** to generate and export datasets.

In [2]:
import os
import json
import re
import tempfile
import pandas as pd
import gradio as gr

from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print(f"API key loaded: {openai_api_key[:8]}...")
else:
    print("OpenRouter/OpenAI API key not found")

API key loaded: sk-or-v1...


In [3]:
openrouter_client = OpenAI(base_url="https://openrouter.ai/api/v1")
ollama_client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

MODELS = {
    "GPT-4.1-mini (OpenRouter)": {
        "client": openrouter_client,
        "model_id": "gpt-4.1-mini",
        "type": "closed-source",
    },
    "Llama 3.2 (Ollama)": {
        "client": ollama_client,
        "model_id": "llama3.2",
        "type": "open-source",
    },
    "DeepSeek-R1 (Ollama)": {
        "client": ollama_client,
        "model_id": "deepseek-r1",
        "type": "open-source",
    },
}

print("Models registered:")
for m in MODELS:
    print("-", m)

Models registered:
- GPT-4.1-mini (OpenRouter)
- Llama 3.2 (Ollama)
- DeepSeek-R1 (Ollama)


In [4]:
PROPERTY_TYPES = [
    "Apartment",
    "Single Family House",
    "Commercial",
    "Mixed Use",
]

RISK_LEVELS = ["Low", "Medium", "High"]

ZONING_TYPES = [
    "Residential",
    "Commercial",
    "Industrial",
    "Mixed",
]

PROPERTY_SCHEMA = {
    "property_id": "unique integer starting at 1",
    "city": "realistic global city",
    "property_type": f"One of: {', '.join(PROPERTY_TYPES)}",
    "price_usd": "estimated property price in USD",
    "year_built": "construction year",
    "size_sqft": "property size in square feet",
    "bedrooms": "number of bedrooms",
    "bathrooms": "number of bathrooms",
    "zoning_type": f"One of: {', '.join(ZONING_TYPES)}",
    "neighborhood_score": "1-10 rating",
    "estimated_rent": "estimated monthly rent USD",
    "risk_level": f"One of: {', '.join(RISK_LEVELS)}",
    "due_diligence_summary": "short professional property analysis",
}

SCHEMA_BLOCK = json.dumps(PROPERTY_SCHEMA, indent=2)

print("Property dataset schema ready")

Property dataset schema ready


In [5]:
def strategy_direct(n):
    system = "You generate synthetic real-estate datasets. Output ONLY JSON."
    
    user = f"""
Generate exactly {n} property due-diligence records as JSON.

Each object must follow this schema:

{SCHEMA_BLOCK}

Create realistic property investment data.
Return ONLY a JSON array.
"""
    return system, user


def strategy_fewshot(n):

    example = """
[
{
"property_id":1,
"city":"Austin",
"property_type":"Apartment",
"price_usd":420000,
"year_built":2015,
"size_sqft":1400,
"bedrooms":3,
"bathrooms":2,
"zoning_type":"Residential",
"neighborhood_score":8,
"estimated_rent":2600,
"risk_level":"Medium",
"due_diligence_summary":"Property located in a growing tech corridor with strong rental demand."
}
]
"""

    system = "You generate synthetic real-estate datasets."

    user = f"""
Example property record:

{example}

Generate {n} NEW property due-diligence records in the same format.
Return ONLY JSON.
"""

    return system, user


def strategy_persona(n):

    system = """
You are a senior real-estate investment analyst writing due-diligence reports.
Return only JSON arrays.
"""

    user = f"""
Generate {n} property investment analyses.

Follow this schema:

{SCHEMA_BLOCK}

Write professional risk analysis in the due_diligence_summary field.
"""

    return system, user


def strategy_market(n):

    system = "You simulate property investment datasets."

    user = f"""
The real estate market currently has:

- rising interest rates
- urban housing shortage
- declining office demand

Generate {n} property due-diligence records reflecting these market conditions.

Use schema:

{SCHEMA_BLOCK}

Return only JSON.
"""

    return system, user


PROMPT_STRATEGIES = {
    "Structured": strategy_direct,
    "Few-Shot": strategy_fewshot,
    "Investor Persona": strategy_persona,
    "Market Scenario": strategy_market,
}

print("Prompt strategies loaded")

Prompt strategies loaded


In [6]:
def strategy_direct(n):
    system = "You generate synthetic real-estate datasets. Output ONLY JSON."
    
    user = f"""
Generate exactly {n} property due-diligence records as JSON.

Each object must follow this schema:

{SCHEMA_BLOCK}

Create realistic property investment data.
Return ONLY a JSON array.
"""
    return system, user


def strategy_fewshot(n):

    example = """
[
{
"property_id":1,
"city":"Austin",
"property_type":"Apartment",
"price_usd":420000,
"year_built":2015,
"size_sqft":1400,
"bedrooms":3,
"bathrooms":2,
"zoning_type":"Residential",
"neighborhood_score":8,
"estimated_rent":2600,
"risk_level":"Medium",
"due_diligence_summary":"Property located in a growing tech corridor with strong rental demand."
}
]
"""

    system = "You generate synthetic real-estate datasets."

    user = f"""
Example property record:

{example}

Generate {n} NEW property due-diligence records in the same format.
Return ONLY JSON.
"""

    return system, user


def strategy_persona(n):

    system = """
You are a senior real-estate investment analyst writing due-diligence reports.
Return only JSON arrays.
"""

    user = f"""
Generate {n} property investment analyses.

Follow this schema:

{SCHEMA_BLOCK}

Write professional risk analysis in the due_diligence_summary field.
"""

    return system, user


def strategy_market(n):

    system = "You simulate property investment datasets."

    user = f"""
The real estate market currently has:

- rising interest rates
- urban housing shortage
- declining office demand

Generate {n} property due-diligence records reflecting these market conditions.

Use schema:

{SCHEMA_BLOCK}

Return only JSON.
"""

    return system, user


PROMPT_STRATEGIES = {
    "Structured": strategy_direct,
    "Few-Shot": strategy_fewshot,
    "Investor Persona": strategy_persona,
    "Market Scenario": strategy_market,
}

print("Prompt strategies loaded")

Prompt strategies loaded


In [7]:
def parse_json_response(text):

    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    text = re.sub(r"```(?:json)?", "", text)
    text = re.sub(r"```", "", text)

    text = text.strip()

    try:
        return json.loads(text)
    except:
        pass

    match = re.search(r"\[.*\]", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except:
            pass

    return None

In [8]:
def generate_properties(model_name, strategy_name, n):

    model_info = MODELS[model_name]
    strategy_fn = PROMPT_STRATEGIES[strategy_name]

    system_msg, user_msg = strategy_fn(n)

    try:

        response = model_info["client"].chat.completions.create(
            model=model_info["model_id"],
            messages=[
                {"role":"system","content":system_msg},
                {"role":"user","content":user_msg}
            ],
            temperature=0.9
        )

        raw = response.choices[0].message.content

    except Exception as e:

        return pd.DataFrame(), str(e)

    records = parse_json_response(raw)

    if records is None:
        return pd.DataFrame(), "JSON parse failed"

    df = pd.DataFrame(records)

    df["model"] = model_name
    df["strategy"] = strategy_name

    return df, f"{model_name} generated {len(df)} properties"

In [9]:
def generate_batch(models, strategy, n):

    all_dfs = []
    status = []

    for model in models:

        df, s = generate_properties(model, strategy, n)

        status.append(s)

        if not df.empty:
            all_dfs.append(df)

    if not all_dfs:
        return pd.DataFrame(), "\n".join(status)

    combined = pd.concat(all_dfs, ignore_index=True)

    combined["property_id"] = range(1, len(combined)+1)

    return combined, "\n".join(status)

In [10]:
def export_csv(df):

    path = os.path.join(tempfile.gettempdir(),"properties.csv")
    df.to_csv(path,index=False)

    return path


def export_json(df):

    path = os.path.join(tempfile.gettempdir(),"properties.json")
    df.to_json(path, orient="records", indent=2)

    return path

In [11]:
with gr.Blocks(title="AI Property Due-Diligence Analyzer") as demo:

    latest_records = gr.State([])

    gr.Markdown("# AI Property Due-Diligence Dataset Generator")

    with gr.Row():

        with gr.Column(scale=1):

            model_select = gr.CheckboxGroup(
                choices=list(MODELS.keys()),
                value=list(MODELS.keys()),
                label="Models"
            )

            strategy_select = gr.Dropdown(
                choices=list(PROMPT_STRATEGIES.keys()),
                value="Structured",
                label="Prompt Strategy"
            )

            num_records = gr.Slider(
                1,20,value=5,step=1,
                label="Properties per model"
            )

            generate_btn = gr.Button("Generate Dataset")

        with gr.Column(scale=2):

            status_box = gr.Textbox(label="Status")

            output_table = gr.Dataframe(label="Generated Properties")

            with gr.Row():

                csv_btn = gr.Button("Download CSV")
                json_btn = gr.Button("Download JSON")

            file_output = gr.File()

    def run_generation(models,strategy,n):

        df,status = generate_batch(models,strategy,int(n))

        records = df.to_dict("records") if not df.empty else []

        return df,status,records

    def run_csv(records):

        if not records:
            return None

        return export_csv(pd.DataFrame(records))

    def run_json(records):

        if not records:
            return None

        return export_json(pd.DataFrame(records))

    generate_btn.click(
        run_generation,
        inputs=[model_select,strategy_select,num_records],
        outputs=[output_table,status_box,latest_records]
    )

    csv_btn.click(run_csv,inputs=[latest_records],outputs=[file_output])
    json_btn.click(run_json,inputs=[latest_records],outputs=[file_output])

demo.launch(inbrowser=True)

Matplotlib is building the font cache; this may take a moment.


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
